In [209]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## Configure dependencies access

In [210]:
BASE_PATH = Path.cwd().parent.parent
ENV_PATH = BASE_PATH / "hospital_production" / ".env"

load_dotenv(ENV_PATH)

# PostgreSQL connection parameters
POSTGRES_USER: str = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD: str = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB: str = os.getenv("POSTGRES_DB")
POSTGRES_HOST: str = "localhost"
POSTGRES_PORT: str = os.getenv("POSTGRES_PORT")
POSTGRES_SCHEMA: str = os.getenv("POSTGRES_SCHEMA")
POSTGRES_TABLE: str = os.getenv("POSTGRES_TABLE")

# Build connection string
DATABASE_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

# Create SQLAlchemy engine
engine = create_engine(DATABASE_URL)

# Open connection and execute query
with engine.connect() as conn:
    query = text(f"SELECT * FROM {POSTGRES_SCHEMA}.{POSTGRES_TABLE}")

    result = conn.execute(query)

    # Fetch and view the results into a pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

## Postgresql data readable

In [211]:
pd.set_option('display.max_columns', None)
df.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens,length_of_stay,patient_churn,capacity_pressure,admission_rate_row,refusal_rate_row,staff_attendance_rate
0,PAT-f9c8afa6,Victor Baker,50,2025-07-30,2025-08-13,general_medicine,91,7,5,40,103.0,40.0,63.0,73.0,52.0,28,23.0,5.0,14,1,2.512195,0.388350,0.611650,0.821429
1,PAT-5290be70,Jeffrey Chandler,29,2025-11-01,2025-11-14,emergency,88,11,1,32,76.0,32.0,44.0,67.0,70.0,39,35.0,4.0,13,1,2.303030,0.421053,0.578947,0.897436
2,PAT-003ce690,Larry Dixon,29,2025-01-19,2025-01-21,ICU,60,1,3,20,21.0,20.0,1.0,82.0,89.0,34,0.0,34.0,2,1,1.000000,0.952381,0.047619,0.000000
3,PAT-18f78014,Kenneth Scott,77,2025-06-24,2025-06-26,general_medicine,61,6,4,43,152.0,43.0,109.0,67.0,66.0,28,22.0,6.0,2,1,3.454545,0.282895,0.717105,0.785714
4,PAT-69dc0dc1,April Frost,37,2025-11-15,2025-11-19,general_medicine,60,11,3,37,58.0,37.0,21.0,95.0,63.0,28,0.0,28.0,4,1,1.526316,0.637931,0.362069,0.000000


## Load ML Models from MinIO Bucket

In [212]:
from dotenv import load_dotenv
BASE_PATH = Path.cwd().parent.parent
ENV_PATH = BASE_PATH / "hospital_production" / ".env"
load_dotenv(ENV_PATH)

True

In [213]:
from minio import Minio
from minio.error import S3Error

# Intialize MinIO client
client = Minio(
    "localhost:9000",
    access_key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    secure=False  # Set to True if using HTTPS
)

# Define bucket
bucket_name = "hospital-models"

try:
    objects = client.list_objects(bucket_name, recursive=True)

    print(f"Model found in bucket '{bucket_name}':")
    for obj in objects:
        print(f" - {obj.object_name}")

except S3Error as e:
    print(f"Error occurred: {e}")

Model found in bucket 'hospital-models':
 - manifest/integrity_audit.json
 - manifest/patient_churn.json
 - manifest/risk_level.json
 - manifest/risk_score.json
 - manifest/service_performance.json
 - manifest/staff_staff_attendance_rate.json
 - manifest/staff_staff_morale.json
 - mlflow/1/137ee340884442a3a4709a03addbd778/artifacts/manifest_patient_churn.json
 - mlflow/1/86df9eacc59448e9bf19e80f594256fc/artifacts/manifest_risk_score.json
 - mlflow/1/9dce29aeaa1b4ca989b5eebe3e3ea306/artifacts/manifest_risk_level.json
 - mlflow/1/a583e441f1854ba7bc7999648306dc02/artifacts/manifest_staff_staff_attendance_rate.json
 - mlflow/1/ad552a5926c94c16a52d6ee0bfc2b8bc/artifacts/manifest_staff_staff_morale.json
 - mlflow/1/d8aa349d747c405a8010ff4844039fa9/artifacts/manifest_service_performance.json
 - mlflow/1/eb8b7cc09b85488b830e232387e35045/artifacts/manifest_integrity_audit.json
 - mlflow/1/models/m-02dbdf35d0c345c9a5ec60af384ec796/artifacts/MLmodel
 - mlflow/1/models/m-02dbdf35d0c345c9a5ec60af38

## Retrieve bundle models & manifest JSON

In [214]:
# Retrieve only models/ name in bucket
ml_models = [obj.object_name for obj in client.list_objects(bucket_name, recursive=True) if obj.object_name.startswith("models/")]
ml_models

['models/integrity_audit_bundle.joblib',
 'models/patient_churn_bundle.joblib',
 'models/risk_level_bundle.joblib',
 'models/risk_score_bundle.joblib',
 'models/service_performance_bundle.joblib',
 'models/staff_staff_attendance_rate_bundle.joblib',
 'models/staff_staff_morale_bundle.joblib']

In [215]:
manifest = [obj.object_name for obj in client.list_objects(bucket_name, recursive=True) if obj.object_name.startswith("manifest/")]
manifest

['manifest/integrity_audit.json',
 'manifest/patient_churn.json',
 'manifest/risk_level.json',
 'manifest/risk_score.json',
 'manifest/service_performance.json',
 'manifest/staff_staff_attendance_rate.json',
 'manifest/staff_staff_morale.json']

In [216]:
import json

manifest_paths = [
    "manifest/integrity_audit.json",
    "manifest/patient_churn.json",
    "manifest/risk_level.json",
    "manifest/risk_score.json",
    "manifest/service_performance.json",
    "manifest/staff_staff_attendance_rate.json",
    "manifest/staff_staff_morale.json",
]

loaded_manifests = []

for path in manifest_paths:
    response = client.get_object(bucket_name, path)

    try:
        loaded_manifests.append(
            {
                "path": path,
                "manifest": json.loads(
                    response.read().decode("utf-8")
                ),
            }
        )
    finally:
        response.close()
        response.release_conn()

## Implemented models:
* ### Use PostgreSQL database for fetching the result
* ### Return to A/B Testing
* ### Implementation models to database and return the result

## A/B Testing

In [217]:
df["service"].value_counts()

service
emergency           2694
surgery             2557
general_medicine    2402
ICU                 2347
Name: count, dtype: int64

In [218]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Select comparison groups by respective model names
SERVICE_A = "emergency"
SERVICE_B = "surgery"
SERVICE_C = "general_medicine"
SERVICE_D = "ICU"

# Preprating data for z-test
data = df[df["service"].isin([SERVICE_A, SERVICE_B, SERVICE_C, SERVICE_D])].copy()

# Check whether counts are valid
if (data["patients_request"] < 0).any():
    raise ValueError("Negative patients request found.")

if (data["patients_admitted"] < 0).any():
    raise ValueError("Negative patients admitted found.")

# Aggregate counts data by service
summary = (
    data.groupby("service")
    .agg(
        visitors=("patients_request", "sum"),
        conversions=("patients_admitted", "sum")
    )
)

# Split into A and B groups
A = summary.loc[SERVICE_A]
B = summary.loc[SERVICE_B]
C = summary.loc[SERVICE_C]
D = summary.loc[SERVICE_D]

requests_A, admitted_A = int(A["visitors"]), int(A["conversions"])
requests_B, admitted_B = int(B["visitors"]), int(B["conversions"])
requests_C, admitted_C = int(C["visitors"]), int(C["conversions"])
requests_D, admitted_D = int(D["visitors"]), int(D["conversions"])

### Calculate admission rates + Two-proprotion Z-test

In [219]:
# Calculate admission rates 
rate_A = admitted_A / requests_A
rate_B = admitted_B / requests_B
rate_C = admitted_C / requests_C
rate_D = admitted_D / requests_D

absolute_difference = rate_B - rate_A
relative_lift = absolute_difference / rate_A if rate_A > 0 else np.nan

# Two proportion z-test
counts = np.asarray([admitted_D, admitted_C])
sample_sizes = np.asarray([requests_D, requests_C])

z_stat, p_value = proportions_ztest(counts, sample_sizes, alternative='two-sided')

# Approximate 95% confidence interval for rate difference (B - A)
se = np.sqrt(rate_A * (1 - rate_A) / requests_A) + (rate_B * (1 - rate_B) / requests_B)

ci_lower = absolute_difference - 1.96 * se
ci_upper = absolute_difference + 1.96 * se

print(f"Admission Rate for {SERVICE_A}: {rate_A:.4f}")
print(f"Admission Rate for {SERVICE_B}: {rate_B:.4f}")
print(f"Admission Rate for {SERVICE_C}: {rate_C:.4f}")
print(f"Admission Rate for {SERVICE_D}: {rate_D:.4f}")
print("=" * 55)
print(f"Z-statistic: {z_stat:.4f}, P-value: {p_value:.4f}")
print(f"Confidence interval for the difference in admission rates (B - A) -> Confidence lower: ({ci_lower:.4f}, Confidence upper: {ci_upper:.4f})")

Admission Rate for emergency: 0.1818
Admission Rate for surgery: 0.5507
Admission Rate for general_medicine: 0.2811
Admission Rate for ICU: 0.8772
Z-statistic: 244.8987, P-value: 0.0000
Confidence interval for the difference in admission rates (B - A) -> Confidence lower: (0.3677, Confidence upper: 0.3700)


### Table results

In [220]:
results_df = pd.DataFrame({
    "Metric": [
        "Patient Requests",
        "Patients Admitted",
        "Admission Rate",
        "Absolute Difference",
        "Relative Lift",
        "Z Statistic",
        "P-Value",
        "95% CI (D - C)"
    ],
    SERVICE_A: [
        f"{requests_A:,}",
        f"{admitted_A:,}",
        f"{rate_A:.2%}",
        f"{absolute_difference:+.2%}",
        f"{relative_lift:+.2%}" if np.isfinite(relative_lift) else "N/A",
        f"{z_stat:.4f}",
        f"{p_value:.4f}",
        f"[{ci_lower:.2%}, {ci_upper:.2%}]"
    ],
    SERVICE_B: [
        f"{requests_D:,}",
        f"{admitted_D:,}",
        f"{rate_D:.2%}",
        f"-",
        f"-",
        f"-",
        f"-",
        f"-"
    ]
}).set_index("Metric")

print("-" * 50)
print("Statistical Inference Results:")
print("-" * 50)
display(results_df)

--------------------------------------------------
Statistical Inference Results:
--------------------------------------------------


,emergency,surgery
Metric,,
Patient Requests,"449,635","43,103"
Patients Admitted,"81,763","37,809"
Admission Rate,18.18%,87.72%
Absolute Difference,+36.88%,-
Relative Lift,+202.84%,-
Z Statistic,244.8987,-
P-Value,0.0000,-
95% CI (D - C),"[36.77%, 37.00%]",-


### Statistical interpretation

In [ ]:
# Assume alpha = 0.05
ALPHA = 0.05

print("\n" + "=" * 55)
print("HOSPITAL ADMISSION RATE ANALYSIS")
print("=" * 55)

if p_value < ALPHA:
    print("Statistically significant difference detected")

    if rate_B > rate_A:
        print(f"{SERVICE_B} has the higher observed admission rate")
    else:
        print(f"{SERVICE_A} has the higher observed admission rate")
else:
    print("No statistically significant difference detected")
    print("The observed difference may be consistent xwith sampling variability or random chance.")

print(f"\nP-value: {p_value:.5f}")
print(f"Admission rate difference: {absolute_difference:+.2%}")
print(f"95% CI: [{ci_lower:.2%}, {ci_upper:.2%}]")
print(
    "\nIMPORTANT: This is a comparison between services, "
    "not evidence of a causal treatment effect."
)


HOSPITAL ADMISSION RATE ANALYSIS
Statistically significant difference detected
surgery has the higher observed admission rate

P-value: 0.00000
Admission rate difference: +36.88%
95% CI: [36.77%, 37.00%]

IMPORTANT: This is a comparison between services, not evidence of a causal treatment effect.


## =====================================================

# Impelementation models on database

In [222]:
loaded_manifests

[{'path': 'manifest/integrity_audit.json',
  'manifest': {'concept': 'integrity_audit',
   'task': 'classification',
   'features': ['age',
    'satisfaction',
    'length_of_stay',
    'available_beds',
    'patients_request',
    'patients_admitted',
    'patients_refused',
    'staff_morale',
    'staff_present',
    'total_staff',
    'staff_absent',
    'capacity_pressure',
    'admission_rate_row',
    'refusal_rate_row',
    'staff_attendance_rate',
    'service'],
   'minio_key': 'integrity_audit_bundle',
   'submodels': ['logistic',
    'decision_tree',
    'random_forest',
    'gradient_boosting',
    'xgboost'],
   'metrics': {'logistic': {'accuracy': 0.853,
     'precision': 0.8513794987165066,
     'recall': 0.853,
     'f1': 0.851286849818334,
     'roc_auc': 0.9323126963381051},
    'decision_tree': {'accuracy': 1.0,
     'precision': 1.0,
     'recall': 1.0,
     'f1': 1.0,
     'roc_auc': 1.0},
    'random_forest': {'accuracy': 0.998,
     'precision': 0.99801141226818

In [223]:
for item in loaded_manifests:
    path = item["path"]
    manifest = item["manifest"]

    features = manifest.get("features", {})

    print(f"Manifest Path: {path}")
    print(f"Features: {json.dumps(features, indent=2)}")

Manifest Path: manifest/integrity_audit.json
Features: [
  "age",
  "satisfaction",
  "length_of_stay",
  "available_beds",
  "patients_request",
  "patients_admitted",
  "patients_refused",
  "staff_morale",
  "staff_present",
  "total_staff",
  "staff_absent",
  "capacity_pressure",
  "admission_rate_row",
  "refusal_rate_row",
  "staff_attendance_rate",
  "service"
]
Manifest Path: manifest/patient_churn.json
Features: [
  "age",
  "satisfaction",
  "length_of_stay",
  "available_beds",
  "patients_request",
  "patients_admitted",
  "patients_refused",
  "staff_morale",
  "staff_present",
  "total_staff",
  "staff_absent",
  "capacity_pressure",
  "admission_rate_row",
  "refusal_rate_row",
  "staff_attendance_rate",
  "service"
]
Manifest Path: manifest/risk_level.json
Features: [
  "age",
  "satisfaction",
  "length_of_stay",
  "available_beds",
  "patients_request",
  "patients_admitted",
  "patients_refused",
  "staff_morale",
  "staff_present",
  "total_staff",
  "staff_absent"

## Feature engineering

In [224]:
# Build required column helper if there missing dependency
df["risk_score"] = (
        (df["patients_request"] <= df["patients_request"].mean()).astype(float)
        + (df["patients_refused"] >= df["patients_refused"].mean()).astype(float)
        + (df["patients_admitted"] <= df["patients_admitted"].mean()).astype(float)
        + (df["satisfaction"] <= df["satisfaction"].mean()).astype(float)
    ) / 4.0
df["risk_level"] = pd.cut(df['risk_score'], bins=[-1, 0.25, 0.5, 0.75, 1], labels=['Low', 'Medium', 'High', 'Critical'])
df["service_performance"] = (df["patient_satisfaction"] * df["patients_admitted"] - df["patients_refused"]) / df["patients_request"]
df["staff_absent"] = df["staff_absens"]
df["health_score"] = (
    0.40 * (df["satisfaction"] / 100)
    + 0.20 * (1 - df["patient_churn"])
    + 0.20 * df["admission_rate_row"]
    + 0.20 * df["staff_attendance_rate"]
) * 100

# Avoid inf in service_performance by replacing inf with NaN and then filling NaN with 0
df["service_performance"].replace([np.inf, -np.inf], np.nan, inplace=True)
df['admission_rate_row'].replace([np.inf, -np.inf], np.nan, inplace=True)
df['refusal_rate_row'].replace([np.inf, -np.inf], np.nan, inplace=True)

# Handle missing values in the DataFrame
df["service_performance"].fillna(df["service_performance"].mean(), inplace=True)
df['health_score'].fillna(df['health_score'].mean(), inplace=True)
df['admission_rate_row'].fillna(0, inplace=True)
df['refusal_rate_row'].fillna(0, inplace=True)

In [225]:
if "missing value" in df.isnull().any().to_string():
    print(df.isnull().sum().sum())
    print("Missing values is still present in the DataFrame. Please check the data and handle missing values accordingly.")
else:
    print(f"✅ Data is clean and ready for analysis. No missing values: {df.isnull().sum().sum()}")

✅ Data is clean and ready for analysis. No missing values: 0


## Extract JSON concept

In [226]:
# Copy the DataFrame to avoid modifying the original data
data = df.copy()
data.head()

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens,length_of_stay,patient_churn,capacity_pressure,admission_rate_row,refusal_rate_row,staff_attendance_rate,risk_score,risk_level,service_performance,staff_absent,health_score
0,PAT-f9c8afa6,Victor Baker,50,2025-07-30,2025-08-13,general_medicine,91,7,5,40,103.0,40.0,63.0,73.0,52.0,28,23.0,5.0,14,1,2.512195,0.388350,0.611650,0.821429,0.00,Low,27.737864,5.0,60.595562
1,PAT-5290be70,Jeffrey Chandler,29,2025-11-01,2025-11-14,emergency,88,11,1,32,76.0,32.0,44.0,67.0,70.0,39,35.0,4.0,13,1,2.303030,0.421053,0.578947,0.897436,0.25,Low,27.631579,4.0,61.569771
2,PAT-003ce690,Larry Dixon,29,2025-01-19,2025-01-21,ICU,60,1,3,20,21.0,20.0,1.0,82.0,89.0,34,0.0,34.0,2,1,1.000000,0.952381,0.047619,0.000000,0.75,High,78.047619,34.0,43.047619
3,PAT-18f78014,Kenneth Scott,77,2025-06-24,2025-06-26,general_medicine,61,6,4,43,152.0,43.0,109.0,67.0,66.0,28,22.0,6.0,2,1,3.454545,0.282895,0.717105,0.785714,0.50,Medium,18.236842,6.0,45.772180
4,PAT-69dc0dc1,April Frost,37,2025-11-15,2025-11-19,general_medicine,60,11,3,37,58.0,37.0,21.0,95.0,63.0,28,0.0,28.0,4,1,1.526316,0.637931,0.362069,0.000000,0.50,Medium,60.241379,28.0,36.758621


In [227]:
import os
import io
import joblib
import numpy as np
import pickle
from minio.error import S3Error

def _fallback_joblib_load(raw_bytes: bytes):
    """Load a joblib‑pickled object, falling back to pickle when a BitGenerator error occurs."""
    try:
        return joblib.load(io.BytesIO(raw_bytes))
    except ValueError as err:
        if "BitGenerator" in str(err):
            try:
                obj = pickle.loads(raw_bytes, encoding="latin1")
                if isinstance(obj, dict) and "random_state" in obj:
                    obj.pop("random_state")
                return obj
            except Exception as e2:
                raise RuntimeError(f"Fallback pickle load failed: {e2}") from e2
        raise

def load_bundle_from_minio(minio_key: str):
    """Download a .joblib artefact from MinIO and return the deserialized object."""
    key = minio_key.strip("/")
    if key.endswith(".joblib"):
        key = key[:-7]
    object_path = (
        f"models/{key}.joblib" if not key.startswith("models/") else f"{key}.joblib"
    )
    try:
        response = client.get_object(bucket_name, object_path)
        raw = response.read()
        return _fallback_joblib_load(raw)
    except S3Error as e:
        raise RuntimeError(f"Unable to fetch {object_path} from bucket {bucket_name}: {e}") from e
    finally:
        if "response" in locals():
            response.close()
            response.release_conn()


In [228]:
# Inspect the manifest and one model's metadata
import json

def get_text(key):
    r = client.get_object(bucket_name, key)
    try:
        return r.read().decode()
    finally:
        r.close()
        r.release_conn()

m = json.loads(get_text("manifest/integrity_audit.json"))
print(json.dumps(m, indent=2)[:3000])
print(get_text("mlflow/1/models/m-a0455db339b64121929f3c023e5e6399/artifacts/MLmodel"))

{
  "concept": "integrity_audit",
  "task": "classification",
  "features": [
    "age",
    "satisfaction",
    "length_of_stay",
    "available_beds",
    "patients_request",
    "patients_admitted",
    "patients_refused",
    "staff_morale",
    "staff_present",
    "total_staff",
    "staff_absent",
    "capacity_pressure",
    "admission_rate_row",
    "refusal_rate_row",
    "staff_attendance_rate",
    "service"
  ],
  "minio_key": "integrity_audit_bundle",
  "submodels": [
    "logistic",
    "decision_tree",
    "random_forest",
    "gradient_boosting",
    "xgboost"
  ],
  "metrics": {
    "logistic": {
      "accuracy": 0.853,
      "precision": 0.8513794987165066,
      "recall": 0.853,
      "f1": 0.851286849818334,
      "roc_auc": 0.9323126963381051
    },
    "decision_tree": {
      "accuracy": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "roc_auc": 1.0
    },
    "random_forest": {
      "accuracy": 0.998,
      "precision": 0.998011412268

In [229]:
import skops.io as sio

def load_skops_model(model_id):
    key = f"mlflow/1/models/{model_id}/artifacts/model.skops"
    local = f"/tmp/{model_id}.skops"
    client.fget_object(bucket_name, key, local)
    untrusted = sio.get_untrusted_types(file=local)
    print(model_id, "untrusted types:", untrusted)   # review this list
    return sio.load(local, trusted=untrusted)

pipe = load_skops_model("m-a0455db339b64121929f3c023e5e6399")
print(pipe)
print(pipe.predict(data[m["features"]].head(3)))   # sanity check

m-a0455db339b64121929f3c023e5e6399 untrusted types: ['numpy.dtype', 'sklearn.compose._column_transformer._RemainderColsList', 'xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier']
Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['age', 'satisfaction',
                                                   'length_of_stay',
                                                   'available_beds',
                                                   'patients_request',
                                                   'patients_admitted',
                                            

In [230]:
for obj in client.list_objects(bucket_name, prefix="models/", recursive=True):
    key = obj.object_name
    try:
        o = load_bundle_from_minio(key)
        info = type(o).__name__

        if isinstance(o, np.ndarray):
            info += f" shape: {o.shape} first={o.flat[0]!r}"
        elif isinstance(o, dict):
            info += f" keys={list(o)[:6]}"

    except Exception as e:
        info = f"Error loading: {e}"

    print(f"{key:60s} -> {info}")

models/integrity_audit_bundle.joblib                         -> ndarray shape: (16,) first='age'
models/patient_churn_bundle.joblib                           -> ndarray shape: (16,) first='age'
models/risk_level_bundle.joblib                              -> ndarray shape: (16,) first='age'
models/risk_score_bundle.joblib                              -> dict keys=['models', 'features', 'task']
models/service_performance_bundle.joblib                     -> dict keys=['models', 'features', 'task']
models/staff_staff_attendance_rate_bundle.joblib             -> dict keys=['models', 'features', 'task']
models/staff_staff_morale_bundle.joblib                      -> dict keys=['models', 'features', 'task']


In [231]:
manifest_by_concept = {
    item["manifest"]["concept"]: item["manifest"] for item in loaded_manifests
}

# Example only: replace with the name you copied from Cell A
manifest_by_concept["integrity_audit"]["minio_key"] = "models/integrity_audit_bundle.joblib"

bundles = {
    concept: load_bundle_from_minio(manifest["minio_key"])
    for concept, manifest in manifest_by_concept.items()
}

print("Loaded bundles:", list(bundles.keys()))
print({c: type(b).__name__ for c, b in bundles.items()})

Loaded bundles: ['integrity_audit', 'patient_churn', 'risk_level', 'risk_score', 'service_performance', 'staff_staff_attendance_rate', 'staff_staff_morale']
{'integrity_audit': 'ndarray', 'patient_churn': 'ndarray', 'risk_level': 'ndarray', 'risk_score': 'dict', 'service_performance': 'dict', 'staff_staff_attendance_rate': 'dict', 'staff_staff_morale': 'dict'}


In [232]:
# Confirm the model is true
m = manifest_by_concept["integrity_audit"]
print(f"Minio key: {m['minio_key']}")
print(f"n features: {len(m['features'])}")
print(f"First 16 features match array: {list(bundles['integrity_audit']) == m['features'][:16]}")

# Compare with a concept that loads correctly
for concept, mf in manifest_by_concept.items():
    print(f"{concept:25s} -> {mf['minio_key']:50s} -> {type(bundles[concept])}")

Minio key: models/integrity_audit_bundle.joblib
n features: 16
First 16 features match array: True
integrity_audit           -> models/integrity_audit_bundle.joblib               -> <class 'numpy.ndarray'>
patient_churn             -> patient_churn_bundle                               -> <class 'numpy.ndarray'>
risk_level                -> risk_level_bundle                                  -> <class 'numpy.ndarray'>
risk_score                -> risk_score_bundle                                  -> <class 'dict'>
service_performance       -> service_performance_bundle                         -> <class 'dict'>
staff_staff_attendance_rate -> staff_staff_attendance_rate_bundle                 -> <class 'dict'>
staff_staff_morale        -> staff_staff_morale_bundle                          -> <class 'dict'>


In [233]:
def normalize_bundle(bundle, concept):
    """Return a plain dict from a bundle that may be a dict or a NumPy‑wrapped dict."""
    
    if isinstance(bundle, dict):
        return bundle

    if isinstance(bundle, np.ndarray) and bundle.size == 1:
        bundle = bundle.reshape(-1)[0]
        if isinstance(bundle, dict):
            return bundle

    hint = ""
    if isinstance(bundle, np.ndarray):
        hint = (f" (array shape={bundle.shape}, first item={bundle.flat[0]!r}). "
                f"This looks like the wrong artefact, so check manifest['minio_key'].")
    raise TypeError(f"{concept}: expected dict bundle, got {type(bundle)}{hint}")

def select_best_model(concept):
    bundle = normalize_bundle(bundles[concept], concept)
    manifest = manifest_by_concept[concept]

    metric = "roc_auc" if manifest["task"] == "classification" else "r2"

    submodel = max(
        manifest["submodels"],
        key=lambda name: manifest["metrics"][name].get(metric, -np.inf),
    )

    models_dict = bundle.get("models") or {
        n: v for n, v in bundle.items() if n in manifest["submodels"]
    }
    if not models_dict:
        raise KeyError(f"No submodels found for concept '{concept}'.")

    if submodel not in models_dict:
        submodel = next(iter(models_dict))

    entry = models_dict[submodel]
    if isinstance(entry, dict):
        pipeline = entry.get("pipeline")
        label_encoder = entry.get("label_encoder", bundle.get("label_encoder"))
    else:
        pipeline = entry
        label_encoder = bundle.get("label_encoder")

    if pipeline is None:
        raise KeyError(f"Pipeline not found for '{concept}/{submodel}'.")

    return submodel, pipeline, label_encoder   # 3 values, matches Cell 5

In [234]:
raw= bundles.get('integrity_audit')
print(f"Raw type: {type(raw)}")
if isinstance(raw, np.ndarray):
    print(f"ndim: {raw.ndim}")
    print(f"Shape: {raw.shape}")
    print(f"dtype: {raw.dtype}")

    try:
        print(f'first element: {type(raw.flat[0]), raw.flat[0]}')
    except Exception as e:
        print(f"Error accessing first element: {e}")

else:
    print(f"Raw content: {raw}")

Raw type: <class 'numpy.ndarray'>
ndim: 1
Shape: (16,)
dtype: object
first element: (<class 'str'>, 'age')


In [235]:
predictions = data[["patient_id"]].copy()

for concept, manifest in manifest_by_concept.items():
    submodel, model, label_encoder = select_best_model(concept)

    features = manifest["features"]

    missing_features = [
        feature
        for feature in features
        if feature not in data.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing features for {concept}: {missing_features}"
        )

    X = (
        data[features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    raw_predictions = model.predict(X)

    if label_encoder is not None:
        try:
            raw_predictions = label_encoder.inverse_transform(raw_predictions)
        except Exception:
            pass

    predictions[f"{concept}_prediction"] = raw_predictions

    if manifest["task"] == "classification" and hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)

        if probabilities.ndim == 2 and probabilities.shape[1] == 2:
            predictions[f"{concept}_probability"] = probabilities[:, 1]
        else:
            predictions[f"{concept}_probability"] = np.asarray(probabilities).max(
                axis=1
            )

    print(f"{concept}: {submodel}")

TypeError: integrity_audit: expected dict bundle, got <class 'numpy.ndarray'> (array shape=(16,), first item='age'). This looks like the wrong artefact, so check manifest['minio_key'].

### Addition business column

In [ ]:
predictions["churn_prediction"] = predictions["patient_churn_predictions"].astype(int)
predictions["high_risk"] = predictions["risk_level_prediction"].astype(str).str.lower().eq("high")
predictions["audit_flag"] = predictions["integrity_audit_predictions"].astype(int)

KeyError: 'patient_churn_predictions'

In [ ]:
# Business summary

summary = pd.DataFrame(
    {
        "metric": [
            "Predicted churn rate",
            "High-risk patients",
            "Average risk score",
            "Average service performance",
            "Average staff morale",
            "Average staff attendance",
            "Integrity audit flags",
        ],
        "value": [
            predictions["churn_prediction"].mean(),
            predictions["high_risk"].sum(),
            predictions["risk_score_prediction"].mean(),
            predictions["service_performance_prediction"].mean(),
            predictions["staff_staff_morale_prediction"].mean(),
            predictions[
                "staff_staff_attendance_rate_prediction"
            ].mean(),
            predictions["audit_flag"].sum(),
        ],
    }
)

summary

KeyError: 'churn_prediction'